# 02times: fastest train times between the 32 cities, 2025 and 2026

**What it does:** for one normal weekday in each timetable, finds the fastest scheduled journey between every pair of the 32 cities (both directions, 992 pairs), with changes of train allowed. "Fastest" means the shortest time from leaving the first station to arriving at the last, over all departures of the day; waiting during a change counts, waiting before the start does not. The search is a profile version of RAPTOR (a standard timetable routing method): it runs once per departure from the origin, latest first, and keeps the best arrival found at every station.

- **2025:** archived Renfe timetable of 18 Dec 2025, Wednesday 14 Jan 2026 (four days before the Adamuz accident).
- **2026:** today's Renfe timetable, Wednesday 25 Nov 2026 (with the temporary speed restrictions set by Adif after the accident).

**Input:** `data/raw/renfe/<version>/` (from `00renfegtfs`), `data/clean/cities.csv` (from `01cities`).

**Output:** for each era `data/clean/times_<era>.csv` (32 x 32 table in minutes, rows = from, columns = to) and `data/clean/journeys_<era>.csv` (one row per pair: minutes, departure, arrival, changes, trains, where to change).

**Data:** Renfe Viajeros, "Horarios de alta velocidad, larga distancia y media distancia", Renfe open data portal (data.renfe.com); archived copy from the Mobility Database (MobilityData), feed mdb-2620.

## 0. Settings

In [1]:
from pathlib import Path
import datetime
import time
import numpy as np
import pandas as pd

RAW_DIR = Path("data") / "raw" / "renfe"  # <- folder of the Renfe versions
CITIES_CSV = Path("data") / "clean" / "cities.csv"  # <- from 01cities
OUT_DIR = Path("data") / "clean"  # <- where the tables go
ERAS = {}
ERAS["2025"] = ("2025-12-18", datetime.date(2026, 1, 14))  # <- (timetable version, day): before the accident
ERAS["2026"] = ("2026-11-16", datetime.date(2026, 11, 25))  # <- (timetable version, day): today
SAME_STATION_CHANGE = 15  # <- minutes needed to change trains at the same station
CROSS_STATION_CHANGE = {"Madrid": 40}  # <- minutes to change between a city's stations (Atocha and Chamartin)
OTHER_CROSS_STATION_CHANGE = 30  # <- the same for the other cities with two stations
MAX_TRAINS = 3  # <- at most 3 trains, that is 2 changes

# Published fastest times (minutes) to compare with. Source: spain.info "Explore Spain on its high-speed trains" (times "from"), and ARA (5 Feb 2026) for Madrid to Barcelona before the restrictions.
REFERENCE = {}
REFERENCE[("Madrid", "Barcelona")] = 157  # <- 2 h 37 before the restrictions (ARA)
REFERENCE[("Madrid", "Sevilla")] = 150  # <- from 2 h 30 (spain.info)
REFERENCE[("Madrid", "Málaga")] = 160  # <- from 2 h 40 (spain.info)
REFERENCE[("Madrid", "Valencia")] = 100  # <- about 100 minutes (spain.info)
REFERENCE[("Barcelona", "Sevilla")] = 318  # <- from 5 h 18 (spain.info)
REFERENCE[("Barcelona", "Málaga")] = 360  # <- from 6 h (spain.info)
REFERENCE[("Valencia", "Sevilla")] = 263  # <- from 4 h 23 (spain.info)

## 1. Helpers: read a timetable and keep one day

In [2]:
def read_gtfs(folder, name):
    table = pd.read_csv(folder / name, dtype=str, keep_default_na=False, encoding="utf-8-sig", skipinitialspace=True)
    table.columns = [c.strip() for c in table.columns]
    for column in table.columns:
        table[column] = table[column].str.strip()
    return table

def to_minutes(text):
    h, m, s = text.split(":")
    return int(h) * 60 + int(m) + int(s) / 60

def clock(minutes):
    m = int(round(minutes))
    return f"{m // 60:02d}:{m % 60:02d}"

def load_day(version, service_date):
    folder = RAW_DIR / version
    calendar = read_gtfs(folder, "calendar.txt")
    calendar_dates = read_gtfs(folder, "calendar_dates.txt")
    trips = read_gtfs(folder, "trips.txt")
    routes = read_gtfs(folder, "routes.txt")
    stop_times = read_gtfs(folder, "stop_times.txt")
    stops = read_gtfs(folder, "stops.txt").set_index("stop_id")
    day = service_date.strftime("%Y%m%d")
    weekday = service_date.strftime("%A").lower()
    running = calendar[(calendar[weekday] == "1") & (calendar["start_date"] <= day) & (calendar["end_date"] >= day)]
    services = set(running["service_id"])
    removed = set(calendar_dates.loc[(calendar_dates["date"] == day) & (calendar_dates["exception_type"] == "2"), "service_id"])
    added = set(calendar_dates.loc[(calendar_dates["date"] == day) & (calendar_dates["exception_type"] == "1"), "service_id"])
    services = (services - removed) | added
    day_trips = trips[trips["service_id"].isin(services)].merge(routes[["route_id", "route_short_name"]], on="route_id", how="left")
    st = stop_times[stop_times["trip_id"].isin(set(day_trips["trip_id"]))].copy()
    st["seq"] = st["stop_sequence"].astype(int)
    st["arr"] = st["arrival_time"].map(to_minutes)
    st["dep"] = st["departure_time"].map(to_minutes)
    st = st.sort_values(["trip_id", "seq"])
    print(f"{version} {service_date} ({weekday}): {len(day_trips)} trips run that day")
    return day_trips, st, stops

## 2. Helper: the profile search

In [3]:
def build_network(day_trips, st, cities):
    stop_list = sorted(st["stop_id"].unique())
    stop_index = {s: i for i, s in enumerate(stop_list)}
    trip_label = dict(zip(day_trips["trip_id"], day_trips["route_short_name"] + " " + day_trips["trip_short_name"]))
    net = {"stop_list": stop_list, "trip_stops": [], "trip_arr": [], "trip_dep": [], "trip_name": []}
    for trip_id, group in st.groupby("trip_id", sort=False):
        net["trip_stops"].append(np.array([stop_index[s] for s in group["stop_id"]]))
        net["trip_arr"].append(group["arr"].to_numpy())
        net["trip_dep"].append(group["dep"].to_numpy())
        net["trip_name"].append(trip_label[trip_id])
    net["serving"] = [[] for _ in stop_list]
    for t, seq_stops in enumerate(net["trip_stops"]):
        for pos, s in enumerate(seq_stops):
            net["serving"][s].append((t, pos))
    net["city_stops"] = []
    missing = []
    for name, ids in zip(cities["city"], cities["stop_ids"]):
        found = [stop_index[s] for s in ids.split(";") if s in stop_index]
        net["city_stops"].append(found)
        if not found:
            missing.append(name)
    net["walks"] = [[] for _ in stop_list]
    for name, group in zip(cities["city"], net["city_stops"]):
        minutes = CROSS_STATION_CHANGE.get(name, OTHER_CROSS_STATION_CHANGE)
        for a in group:
            for b in group:
                if a != b:
                    net["walks"][a].append((b, minutes))
    print("  cities with no station served that day:", missing if missing else "none")
    return net

def profile_from(net, origin, n_cities):
    INF = 1e9
    trip_stops = net["trip_stops"]
    trip_arr = net["trip_arr"]
    trip_dep = net["trip_dep"]
    serving = net["serving"]
    starts = net["city_stops"][origin]
    n_stops = len(net["stop_list"])
    departures = set()
    for s in starts:
        for t, pos in serving[s]:
            if pos < len(trip_stops[t]) - 1:
                departures.add(trip_dep[t][pos])
    tau = np.full((MAX_TRAINS + 1, n_stops), INF)
    parent = [dict() for _ in range(MAX_TRAINS + 1)]
    best = {}
    for t_start in sorted(departures, reverse=True):
        marked = set(starts)
        for k in range(1, MAX_TRAINS + 1):
            boarding = {}
            for s in marked:
                if k == 1:
                    options = [(t_start, s)]
                else:
                    options = [(tau[k - 1][s] + SAME_STATION_CHANGE, s)]
                    for target, minutes in net["walks"][s]:
                        options.append((tau[k - 1][s] + minutes, target))
                for ready, at in options:
                    for t, pos in serving[at]:
                        if trip_dep[t][pos] >= ready and pos < len(trip_stops[t]) - 1:
                            if t not in boarding or pos < boarding[t][0]:
                                boarding[t] = (pos, s)
            new_marked = set()
            for t, (pos, from_stop) in boarding.items():
                for p in range(pos + 1, len(trip_stops[t])):
                    s2 = trip_stops[t][p]
                    if trip_arr[t][p] < tau[k][s2]:
                        tau[k][s2] = trip_arr[t][p]
                        parent[k][s2] = (t, pos, p, from_stop)
                        new_marked.add(s2)
            marked = new_marked
            if not marked:
                break
        for dest in range(n_cities):
            if dest == origin:
                continue
            for k in range(1, MAX_TRAINS + 1):
                for s in net["city_stops"][dest]:
                    if tau[k][s] >= INF:
                        continue
                    legs = []
                    node = s
                    for kk in range(k, 0, -1):
                        t, pos, p, from_stop = parent[kk][node]
                        legs.append((t, pos, p))
                        node = from_stop
                    legs.reverse()
                    depart = trip_dep[legs[0][0]][legs[0][1]]
                    duration = tau[k][s] - depart
                    if dest not in best or duration < best[dest][0] - 1e-9:
                        best[dest] = (duration, depart, tau[k][s], legs)
    return best

## 3. Run both eras and save the tables

In [4]:
cities = pd.read_csv(CITIES_CSV, dtype={"main_stop_id": str, "stop_ids": str})
names = list(cities["city"])
journeys = {}
for era, (version, service_date) in ERAS.items():
    t0 = time.time()
    day_trips, st, stops = load_day(version, service_date)
    net = build_network(day_trips, st, cities)
    rows = []
    for o in range(len(names)):
        best = profile_from(net, o, len(names))
        for d, (duration, depart, arrive, legs) in best.items():
            trains = " + ".join(net["trip_name"][t] for t, _, _ in legs)
            change_at = " / ".join(stops.loc[net["stop_list"][net["trip_stops"][t][p]], "stop_name"] for t, _, p in legs[:-1])
            rows.append({"origin": names[o], "destination": names[d], "minutes": round(duration, 1), "depart": clock(depart), "arrive": clock(arrive), "changes": len(legs) - 1, "trains": trains, "change_at": change_at})
    table = pd.DataFrame(rows)
    journeys[era] = table
    matrix = table.pivot(index="origin", columns="destination", values="minutes").reindex(index=names, columns=names)
    table.to_csv(OUT_DIR / f"journeys_{era}.csv", index=False, encoding="utf-8")
    matrix.to_csv(OUT_DIR / f"times_{era}.csv", encoding="utf-8")
    print(f"  {era}: {len(table)} of {len(names) * (len(names) - 1)} pairs connected, {time.time() - t0:.1f} s; saved times_{era}.csv and journeys_{era}.csv")

2025-12-18 2026-01-14 (wednesday): 1446 trips run that day
  cities with no station served that day: none
  2025: 992 of 992 pairs connected, 2.8 s; saved times_2025.csv and journeys_2025.csv
2026-11-16 2026-11-25 (wednesday): 1610 trips run that day
  cities with no station served that day: none
  2026: 992 of 992 pairs connected, 2.7 s; saved times_2026.csv and journeys_2026.csv


## 4. Check against published times

In [5]:
for (a, b), ref in REFERENCE.items():
    line = f"{a:10s} to {b:10s} published {ref:4d} min"
    for era, table in journeys.items():
        row = table[(table["origin"] == a) & (table["destination"] == b)].iloc[0]
        line += f" | {era}: {row['minutes']:5.0f} min ({row['trains']})"
    print(line)

Madrid     to Barcelona  published  157 min | 2025:   157 min (AVE 03191) | 2026:   182 min (AVE 03191)
Madrid     to Sevilla    published  150 min | 2025:   159 min (AVE 02190) | 2026:   158 min (AVE 02160)
Madrid     to Málaga     published  160 min | 2025:   159 min (AVE 02172) | 2026:   164 min (AVE 02172)
Madrid     to Valencia   published  100 min | 2025:   114 min (AVE 05200) | 2026:   114 min (AVE 05200)
Barcelona  to Sevilla    published  318 min | 2025:   340 min (AVE 03070 + AVE 02100) | 2026:   374 min (AVE 03150 + AVE 02190)
Barcelona  to Málaga     published  360 min | 2025:   354 min (AVE 03080 + AVE 03882) | 2026:   380 min (AVE 03160 + AVE 02192)
Valencia   to Sevilla    published  263 min | 2025:   260 min (AVE 03982) | 2026:   260 min (AVE 03982)


**How to read the check:** the 2025 times should sit close to the published ones (Madrid to Barcelona 2 h 37). Published "from" times can be a few minutes faster than Renfe's own trains when they come from another operator or an older timetable (Madrid to Valencia, about 100 minutes, is one such case). The 2026 times include the speed restrictions, so several are slower.

## 5. From Madrid in both eras

In [6]:
madrid = journeys["2025"][journeys["2025"]["origin"] == "Madrid"][["destination", "minutes", "changes", "trains"]].rename(columns={"minutes": "min_2025", "changes": "chg_2025", "trains": "trains_2025"})
madrid_2026 = journeys["2026"][journeys["2026"]["origin"] == "Madrid"][["destination", "minutes", "changes"]].rename(columns={"minutes": "min_2026", "changes": "chg_2026"})
madrid = madrid.merge(madrid_2026, on="destination")
madrid["change"] = madrid["min_2026"] - madrid["min_2025"]
print(madrid.sort_values("min_2025").to_string(index=False))

  destination  min_2025  chg_2025          trains_2025  min_2026  chg_2026  change
  Ciudad Real      54.0         0            AVE 02210      53.0         0    -1.0
   Valladolid      54.0         0            AVE 05171      54.0         0     0.0
     Zaragoza      78.0         0            AVE 03183      95.0         0    17.0
     Albacete      90.0         0            AVE 02913      90.0         0     0.0
      Córdoba     104.0         0            AVE 02196     104.0         0     0.0
     Valencia     114.0         0            AVE 05200     114.0         0     0.0
         León     120.0         0            AVE 05183     120.0         0     0.0
       Lleida     122.0         0            AVE 02591     143.0         0    21.0
    Tarragona     124.0         0            AVE 03091     147.0         0    23.0
      Ourense     134.0         0            AVE 05273     134.0         0     0.0
     Alicante     143.0         0            AVE 05182     143.0         0     0.0
    

## 6. How much did the whole network change from 2025 to 2026?

In [7]:
both = journeys["2025"].merge(journeys["2026"], on=["origin", "destination"], suffixes=("_2025", "_2026"))
both["change"] = both["minutes_2026"] - both["minutes_2025"]
print("Median change, all pairs:", both["change"].median(), "min")
print("Pairs at least 10 min slower:", int((both["change"] >= 10).sum()), "of", len(both))
print("Pairs at least 10 min faster:", int((both["change"] <= -10).sum()), "of", len(both))
print()
print("Largest changes (both directions listed separately):")
print(both.reindex(both["change"].abs().sort_values(ascending=False).index).head(10)[["origin", "destination", "minutes_2025", "minutes_2026", "change", "changes_2025", "changes_2026"]].to_string(index=False))

Median change, all pairs: 4.0 min
Pairs at least 10 min slower: 372 of 992
Pairs at least 10 min faster: 108 of 992

Largest changes (both directions listed separately):
       origin destination  minutes_2025  minutes_2026  change  changes_2025  changes_2026
    Santander     Badajoz         696.0         557.0  -139.0             2             1
      Sevilla     Badajoz         389.0         260.0  -129.0             1             1
       Huelva     Badajoz         485.0         610.0   125.0             2             2
       Teruel      Oviedo         524.0         648.0   124.0             2             2
San Sebastián     Granada         596.0         692.0    96.0             2             1
       Bilbao        Jaén         677.0         585.0   -92.0             2             2
         Jaén      Girona         470.0         557.0    87.0             2             2
    Tarragona     Badajoz         502.0         586.0    84.0             1             2
        Cádiz     Ba

**Note:** long journeys with two changes depend on how connections happen to line up that day, so they can swing by an hour or more between timetables for reasons that have nothing to do with speed. The direct lines from Madrid are the cleaner measure of the slowdown.

___
**END**